# Reproduce the paper's evaluation

This notebook reproduces the fork-detection accuracy figures and the base-trunk height RMSE against the MLS reference reported in the manuscript.

## How to run it

Three levels of reproducibility, depending on what data you have available.

1. **Reading only.** All cells below have been executed and their outputs are checked in, so the tables and figures are visible on GitHub without running anything.

2. **From cached records.** The per-tree records are shipped as `results/*.pkl` (a few tens of KB each). Restart the kernel and Run All to regenerate the tables and figures from those cached records. No dataset access needed.

3. **From scratch.** Set `EVO_DATA_ROOT` to the dataset root, delete `results/*.pkl`, then Run All. The per-tree records are recomputed from the skeleton PLYs. Requires the ALS/MLS data (currently restricted; see the paper's data-availability statement).

In [ ]:
# Setup: add the reproduction package to sys.path
import sys
from pathlib import Path

PAPER_DIR = Path.cwd()
sys.path.insert(0, str(PAPER_DIR))
sys.path.insert(0, str(PAPER_DIR / 'src'))

import config
from runner import run_dataset
from metrics import (fork_detection_table, print_fork_detection,
                     funnel_row, print_funnel_row,
                     pair_estimates_with_reference,
                     height_metrics, print_height_metrics)

## Fork detection

Three evaluation settings are reported. `run_dataset` uses cached records from `results/*.pkl` when available; pass `force_rerun=True` to recompute from the skeletons.

In [ ]:
def _table(forked_ds, unforked_ds):
    return fork_detection_table(run_dataset(forked_ds) + run_dataset(unforked_ds))

manual_tab = _table('manual_forked', 'manual_unforked')
print_fork_detection(manual_tab, title='Manually corrected (standalone upper bound)')

In [ ]:
PIPELINES = {
    'ForestFormer3D (FOR-instanceV2)':          {
        'inst': ('p1_INST_forked', 'p1_INST_unforked'),
        'sem':  ('p1_SEM_forked',  'p1_SEM_unforked'),
    },
    'ForestFormer3D (FGI-EMIT) + ResNet16FPN':  {
        'inst': ('p2_INST_forked', 'p2_INST_unforked'),
        'sem':  ('p2_SEM_forked',  'p2_SEM_unforked'),
    },
}

for pipeline, stages in PIPELINES.items():
    inst = _table(*stages['inst'])
    sem  = _table(*stages['sem'])
    print(f"\n=== {pipeline} ===")
    for cls, total in (('forked', config.TOTAL_FORKED),
                       ('unforked', config.TOTAL_UNFORKED)):
        print_funnel_row(funnel_row(cls, inst, sem, total), pipeline=pipeline)

## Base-trunk height against the MLS reference

For each setting, restrict to trees that have an MLS reference and were detected as forked, pair the predicted height against the reference, and report RMSE, bias, MAE, R^2 and relative RMSE.

In [ ]:
from reference import load_mls_reference

lookup, mls_heights, _ = load_mls_reference()

SETTINGS = {
    'Manually corrected':                       'manual_forked',
    'ForestFormer3D (FOR-instanceV2)':          'p1_INST_forked',
    'ForestFormer3D (FGI-EMIT) + ResNet16FPN':  'p2_INST_forked',
}

for setting, ds in SETTINGS.items():
    recs = [r for r in run_dataset(ds) if lookup(r['name']) is not None]
    paired = pair_estimates_with_reference(recs, lookup, mls_heights,
                                           height_field='height_minmax')
    print_height_metrics(height_metrics(paired), title=setting)

## Figures

TODO: add the paper's figures (agreement scatter, error distribution) as further cells, driven by the same paired records above.